## Grader Validation
This notebook contains a basic elaboration of the grader validation data.

In [11]:
import pandas as pd
from datasets import load_dataset

In [ ]:
def compute_metrics(human_answers: list[int], model_answers: list[int]) -> dict[str, float]:
	"""
	Compute accuracy, precision, recall, and F1 score for the model's answers compared to human answers.
	Consider human answers as the ground truth and model answers as the predictions.
	"""
	assert len(human_answers) == len(model_answers), "The length of human answers and model answers must be the same."

	tp = sum(1 for h, m in zip(human_answers, model_answers) if h == 1 and m == 1)
	fp = sum(1 for h, m in zip(human_answers, model_answers) if h == 0 and m == 1)
	fn = sum(1 for h, m in zip(human_answers, model_answers) if h == 1 and m == 0)
	tn = sum(1 for h, m in zip(human_answers, model_answers) if h == 0 and m == 0)	
	accuracy = (tp + tn) / len(human_answers) if len(human_answers) > 0 else 0
	precision = tp / (tp + fp) if (tp + fp) > 0 else 0
	recall = tp / (tp + fn) if (tp + fn) > 0 else 0
	f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
	fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
	return {
		"accuracy": accuracy,
		"precision": precision,
		"recall": recall,
		"fpr": fpr,
		"f1_score": f1_score
	}

In [ ]:
df_text = pd.read_csv("../data/validation/text_output_valid.csv")
df_img = pd.read_csv("../data/validation/img_output_valid.csv")

valid_ids = load_dataset("blind-spots-neurips/blind-spots-bench", split="test").to_pandas()["QID"]

df_img = df_img[df_img["id"].isin(valid_ids)]
df_text = df_text[df_text["id"].isin(valid_ids)]

df_text = df_text[df_text["human_and_grader_scores_match"].notna()]

df_img.rename(columns={"verifier_score": "grader_score", "model": "model_name"}, inplace=True)
df_img["grader_score"] = df_img["grader_score"].apply(lambda x: 1 if x == "C" else 0)
df_img["human_score"]= df_img["human_score"].apply(lambda x: 1 if x == "C" else 0)

df_text["grader_score"] = df_text["grader_score"].apply(lambda x: 1 if x else 0)
df_text["human_score"] = df_text["human_score"].apply(lambda x: 1 if x else 0)


### General agreement check

In [14]:

for df, modality in zip([df_text, df_img], ["text-outputs", "image-outputs"]):
	print(modality)
	print(f"\tNumber of samples: {len(df)}")
	print("\tDistribution of grader scores: {} correct, {} incorrect".format(df["grader_score"].sum(), len(df) - df["grader_score"].sum()))
	print("\tDistribution of human scores: {} correct, {} incorrect".format(df["human_score"].sum(), len(df) - df["human_score"].sum()))
	metrics = compute_metrics(df["human_score"].tolist(), df["grader_score"].tolist())
	print("\tMetrics:")
	for metric_name, metric_value in metrics.items():
		print(f"\t\t{metric_name}: {metric_value:.4f}")

text-outputs
	Number of samples: 88
	Distribution of grader scores: 48 correct, 40 incorrect
	Distribution of human scores: 47 correct, 41 incorrect
	Metrics:
		accuracy: 0.9659
		precision: 0.9583
		recall: 0.9787
		fpr: 0.0488
		f1_score: 0.9684
image-outputs
	Number of samples: 30
	Distribution of grader scores: 15 correct, 15 incorrect
	Distribution of human scores: 15 correct, 15 incorrect
	Metrics:
		accuracy: 0.8667
		precision: 0.8667
		recall: 0.8667
		fpr: 0.1333
		f1_score: 0.8667


### Pro-Google bias in grading

In [15]:
for df, modality in zip([df_text, df_img], ["text-outputs", "image-outputs"]):
	print(modality)

	for subset in ["google", "others"]:		
		print(f"\tSubset: {subset}")
		if subset == "google":
			tmp_df = df[(df["model_name"].str.contains("gemini", case=False)) | df["model_name"].str.contains("gemma", case=False)].copy()
		else:
			tmp_df = df[~(df["model_name"].str.contains("gemini", case=False)) & ~df["model_name"].str.contains("gemma", case=False)].copy()
	
		print("\t\tDistribution of grader scores: {} correct, {} incorrect".format(tmp_df["grader_score"].sum(), len(tmp_df) - tmp_df["grader_score"].sum()))
		print("\t\tDistribution of human scores: {} correct, {} incorrect".format(tmp_df["human_score"].sum(), len(tmp_df) - tmp_df["human_score"].sum()))

		print(f"\t\tNumber of samples: {len(tmp_df)}")
		metrics = compute_metrics(tmp_df["human_score"].tolist(), tmp_df["grader_score"].tolist())
		print("\t\tMetrics:")
		for metric_name, metric_value in metrics.items():
			print(f"\t\t\t{metric_name}: {metric_value:.4f}")

text-outputs
	Subset: google
		Distribution of grader scores: 12 correct, 9 incorrect
		Distribution of human scores: 13 correct, 8 incorrect
		Number of samples: 21
		Metrics:
			accuracy: 0.9524
			precision: 1.0000
			recall: 0.9231
			fpr: 0.0000
			f1_score: 0.9600
	Subset: others
		Distribution of grader scores: 36 correct, 31 incorrect
		Distribution of human scores: 34 correct, 33 incorrect
		Number of samples: 67
		Metrics:
			accuracy: 0.9701
			precision: 0.9444
			recall: 1.0000
			fpr: 0.0606
			f1_score: 0.9714
image-outputs
	Subset: google
		Distribution of grader scores: 8 correct, 7 incorrect
		Distribution of human scores: 7 correct, 8 incorrect
		Number of samples: 15
		Metrics:
			accuracy: 0.9333
			precision: 0.8750
			recall: 1.0000
			fpr: 0.1250
			f1_score: 0.9333
	Subset: others
		Distribution of grader scores: 7 correct, 8 incorrect
		Distribution of human scores: 8 correct, 7 incorrect
		Number of samples: 15
		Metrics:
			accuracy: 0.8000
			precision: 0.8